In [ ]:
!pip -q install nflows

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import pickle

from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from nflows.flows.base import Flow
from nflows.distributions.normal import StandardNormal
from nflows.transforms.base import CompositeTransform
from nflows.transforms.permutations import RandomPermutation
from nflows.transforms.lu import LULinear
from nflows.transforms.autoregressive import MaskedPiecewiseRationalQuadraticAutoregressiveTransform


seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_default_dtype(torch.float32)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


TRAIN_CSV = "/content/train.csv"
VAL_CSV   = "/content/val.csv"
OUT_DIR   = "/content/flow_out_inverse"
os.makedirs(OUT_DIR, exist_ok=True)


c_cols = [
    "Frequency (Hz)",
    "Storage modulus (Pa)",
    "Loss modulus (Pa)",
]

x_cols = [
    "Acylamide Conc. %",
    "Bis-acrylamide conc %",
    "Photo-initiator conc. %",
    "Layer Height. (micron)",
    "Bottom Layer exposure time (s) ",
    "Exposure time (s)",
]


def read_csv_clean(path):
    df = pd.read_csv(path)
    df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed")], errors="ignore")
    return df

df_tr = read_csv_clean(TRAIN_CSV)
df_va = read_csv_clean(VAL_CSV)

needed = c_cols + x_cols
df_tr = df_tr.dropna(subset=needed).reset_index(drop=True)
df_va = df_va.dropna(subset=needed).reset_index(drop=True)

C_tr_raw = df_tr[c_cols].to_numpy(np.float32)
X_tr_raw = df_tr[x_cols].to_numpy(np.float32)
C_va_raw = df_va[c_cols].to_numpy(np.float32)
X_va_raw = df_va[x_cols].to_numpy(np.float32)

print("Train rows:", len(df_tr), "| Val rows:", len(df_va))
print("c dim:", C_tr_raw.shape[1], "| x dim:", X_tr_raw.shape[1])


# def make_default_logmask(cols):
#     # default False
#     return np.array([False]*len(cols), dtype=bool)

c_logmask = np.zeros(len(c_cols), dtype=bool)
x_logmask = np.ones(len(x_cols), dtype=bool)

c_logmask[c_cols.index("Frequency (Hz)")] = True
c_logmask[c_cols.index("Storage modulus (Pa)")] = True
c_logmask[c_cols.index("Loss modulus (Pa)")] = True

for name in ["Storage modulus (Pa)", "Loss modulus (Pa)"]:
    if name in c_cols:
        c_logmask[c_cols.index(name)] = True

if "Frequency (Hz)" in c_cols:
    c_logmask[c_cols.index("Frequency (Hz)")] = True

for name in ["Acylamide Conc. %", "Bis-acrylamide conc %", "Photo-initiator conc. %"]:
    if name in x_cols:
        x_logmask[x_cols.index(name)] = True


# for name in ["Layer Height. (micron)", "Bottom Layer exposure time (s) ", "Exposure time (s)"]:
#     if name in x_cols:
#         x_logmask[x_cols.index(name)] = True

print("c_logmask:", dict(zip(c_cols, c_logmask.tolist())))
print("x_logmask:", dict(zip(x_cols, x_logmask.tolist())))

def apply_log1p(arr2d, logmask):
    out = arr2d.astype(np.float32).copy()
    for j in range(out.shape[1]):
        if logmask[j]:
            out[:, j] = np.log1p(np.clip(out[:, j], 0.0, None))
    return out

C_tr_p = apply_log1p(C_tr_raw, c_logmask)
X_tr_p = apply_log1p(X_tr_raw, x_logmask)
C_va_p = apply_log1p(C_va_raw, c_logmask)
X_va_p = apply_log1p(X_va_raw, x_logmask)

c_scaler = StandardScaler()
x_scaler = StandardScaler()
C_tr_n = c_scaler.fit_transform(C_tr_p).astype(np.float32)
X_tr_n = x_scaler.fit_transform(X_tr_p).astype(np.float32)
C_va_n = c_scaler.transform(C_va_p).astype(np.float32)
X_va_n = x_scaler.transform(X_va_p).astype(np.float32)

meta = dict(
    c_cols=c_cols,
    x_cols=x_cols,
    c_logmask=c_logmask,
    x_logmask=x_logmask,
    seed=seed,
)

with open(os.path.join(OUT_DIR, "meta.pkl"), "wb") as f:
    pickle.dump(meta, f)
with open(os.path.join(OUT_DIR, "c_scaler.pkl"), "wb") as f:
    pickle.dump(c_scaler, f)
with open(os.path.join(OUT_DIR, "x_scaler.pkl"), "wb") as f:
    pickle.dump(x_scaler, f)

d_x = X_tr_n.shape[1]
d_c = C_tr_n.shape[1]

num_layers = 6
hidden_features = 256
num_bins = 16
tail_bound = 3.0
dropout_probability = 0.0

min_bin_width  = 1e-3
min_bin_height = 1e-3
min_derivative = 1e-3

transforms = []
for _ in range(num_layers):
    transforms.append(RandomPermutation(features=d_x))
    transforms.append(LULinear(d_x))
    transforms.append(
        MaskedPiecewiseRationalQuadraticAutoregressiveTransform(
            features=d_x,
            hidden_features=hidden_features,
            context_features=d_c,
            num_bins=num_bins,
            tails="linear",
            tail_bound=tail_bound,
            dropout_probability=dropout_probability,
            min_bin_width=min_bin_width,
            min_bin_height=min_bin_height,
            min_derivative=min_derivative,
        )
    )

flow = Flow(CompositeTransform(transforms), StandardNormal([d_x])).to(device).float()

def nll(flow, x, c):
    return -flow.log_prob(inputs=x, context=c).mean()

batch_size = 256
epochs = 100
lr = 3e-4

train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_tr_n, dtype=torch.float32),
        torch.tensor(C_tr_n, dtype=torch.float32),
    ),
    batch_size=batch_size,
    shuffle=True
)

X_va_t = torch.tensor(X_va_n, dtype=torch.float32, device=device)
C_va_t = torch.tensor(C_va_n, dtype=torch.float32, device=device)

opt = torch.optim.AdamW(flow.parameters(), lr=lr, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=3e-5)

best_val = float("inf")
best_state = None
patience = 20
no_improve = 0

for ep in range(1, epochs + 1):
    flow.train()
    tr_losses = []

    for xb, cb in train_loader:
        xb = xb.to(device)
        cb = cb.to(device)
        opt.zero_grad()
        loss = nll(flow, xb, cb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(flow.parameters(), 5.0)
        opt.step()
        tr_losses.append(loss.item())

    flow.eval()
    with torch.no_grad():
        v = nll(flow, X_va_t, C_va_t).item()

    sch.step()
    print(f"Epoch {ep:03d} | lr {opt.param_groups[0]['lr']:.2e} | train NLL {np.mean(tr_losses):.4f} | val NLL {v:.4f}")

    if v < best_val - 1e-4:
        best_val = v
        best_state = {k: p.detach().cpu().clone() for k, p in flow.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping. Best val NLL = {best_val:.4f}")
            break

torch.save(best_state, os.path.join(OUT_DIR, "best_flow_state.pt"))
print("Saved model + scalers to:", OUT_DIR)

  Preparing metadata (setup.py) ... done
Device: cuda
Train rows: 1868 | Val rows: 234
c dim: 3 | x dim: 6
c_logmask: {'Frequency (Hz)': True, 'Storage modulus (Pa)': True, 'Loss modulus (Pa)': True}
x_logmask: {'Acylamide Conc. %': True, 'Bis-acrylamide conc %': True, 'Photo-initiator conc. %': True, 'Layer Height. (micron)': True, 'Bottom Layer exposure time (s) ': True, 'Exposure time (s)': True}
Epoch 001 | lr 3.00e-04 | train NLL 9.6562 | val NLL 6.2180
Epoch 002 | lr 3.00e-04 | train NLL 4.7503 | val NLL 3.5367
Epoch 003 | lr 2.99e-04 | train NLL 2.4780 | val NLL 1.7054
Epoch 004 | lr 2.99e-04 | train NLL 0.7901 | val NLL 1.0130
Epoch 005 | lr 2.98e-04 | train NLL -0.7457 | val NLL -0.0664
Epoch 006 | lr 2.98e-04 | train NLL -1.4412 | val NLL -1.2648
Epoch 007 | lr 2.97e-04 | train NLL -1.9647 | val NLL -1.6678
Epoch 008 | lr 2.96e-04 | train NLL -2.8311 | val NLL -2.8468
Epoch 009 | lr 2.95e-04 | train NLL -4.4075 | val NLL -4.9435
Epoch 010 | lr 2.93e-04 | train NLL -5.6315 | v

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import pickle

from nflows.flows.base import Flow
from nflows.distributions.normal import StandardNormal
from nflows.transforms.base import CompositeTransform
from nflows.transforms.permutations import RandomPermutation
from nflows.transforms.lu import LULinear
from nflows.transforms.autoregressive import MaskedPiecewiseRationalQuadraticAutoregressiveTransform

torch.set_default_dtype(torch.float32)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

TEST_CSV  = "/content/test.csv"
OUT_DIR   = "/content/flow_out_inverse"
OUT_SYNTH = "/content/synth.csv"

META_PATH  = os.path.join(OUT_DIR, "meta.pkl")
C_SCALER_P = os.path.join(OUT_DIR, "c_scaler.pkl")
X_SCALER_P = os.path.join(OUT_DIR, "x_scaler.pkl")
STATE_P    = os.path.join(OUT_DIR, "best_flow_state.pt")

for p in [TEST_CSV, META_PATH, C_SCALER_P, X_SCALER_P, STATE_P]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing file: {p}")

with open(META_PATH, "rb") as f:
    meta = pickle.load(f)
with open(C_SCALER_P, "rb") as f:
    c_scaler = pickle.load(f)
with open(X_SCALER_P, "rb") as f:
    x_scaler = pickle.load(f)

c_cols = meta["c_cols"]
x_cols = meta["x_cols"]
c_logmask = np.array(meta["c_logmask"], dtype=bool)
x_logmask = np.array(meta["x_logmask"], dtype=bool)

d_c = len(c_cols)
d_x = len(x_cols)

print("Condition cols:", c_cols)
print("Generated cols :", x_cols)

def apply_log1p(arr2d, logmask):
    out = arr2d.astype(np.float32).copy()
    for j in range(out.shape[1]):
        if logmask[j]:
            out[:, j] = np.log1p(np.clip(out[:, j], 0.0, None))
    return out

def invert_log1p(arr2d, logmask):
    out = arr2d.astype(np.float32).copy()
    for j in range(out.shape[1]):
        if logmask[j]:
            out[:, j] = np.expm1(out[:, j])
    return out

num_layers = 6
hidden_features = 256
num_bins = 16
tail_bound = 3.0
dropout_probability = 0.0

min_bin_width  = 1e-3
min_bin_height = 1e-3
min_derivative = 1e-3

transforms = []
for _ in range(num_layers):
    transforms.append(RandomPermutation(features=d_x))
    transforms.append(LULinear(d_x))
    transforms.append(
        MaskedPiecewiseRationalQuadraticAutoregressiveTransform(
            features=d_x,
            hidden_features=hidden_features,
            context_features=d_c,
            num_bins=num_bins,
            tails="linear",
            tail_bound=tail_bound,
            dropout_probability=dropout_probability,
            min_bin_width=min_bin_width,
            min_bin_height=min_bin_height,
            min_derivative=min_derivative,
        )
    )

flow = Flow(CompositeTransform(transforms), StandardNormal([d_x])).to(device).float()
state = torch.load(STATE_P, map_location=device)
flow.load_state_dict(state)
flow.eval()

df_test = pd.read_csv(TEST_CSV)
df_test = df_test.drop(columns=[c for c in df_test.columns if str(c).startswith("Unnamed")], errors="ignore")
orig_cols = list(df_test.columns)

for c in c_cols:
    if c not in df_test.columns:
        raise KeyError(f"Test set missing conditioning column: {c}")
for c in x_cols:
    if c not in df_test.columns:
        raise KeyError(f"Test set missing generated column (needed so we can overwrite): {c}")


C_raw = df_test[c_cols].to_numpy(np.float32)
C_p   = apply_log1p(C_raw, c_logmask)
C_n   = c_scaler.transform(C_p).astype(np.float32)

C_t = torch.tensor(C_n, dtype=torch.float32, device=device)

@torch.no_grad()
def sample_one_per_row_manual(flow, C_t, batch_size=1024):
    """
    Samples exactly 1 x per condition row using manual z ~ N(0,I)
    and x = f^{-1}(z; c). This bypasses flow.sample() entirely.
    Returns: (N, d_x) numpy float32 in STANDARDIZED x-space.
    """
    flow.eval()
    N = C_t.shape[0]
    X_out = np.empty((N, d_x), dtype=np.float32)

    i = 0
    while i < N:
        bs = min(batch_size, N - i)
        cb = C_t[i:i+bs]

        z = torch.randn((bs, d_x), device=cb.device, dtype=cb.dtype)

        x, _ = flow._transform.inverse(z, context=cb)

        X_out[i:i+bs] = x.detach().cpu().numpy().astype(np.float32)
        i += bs

    return X_out

X_n = sample_one_per_row_manual(flow, C_t, batch_size=1024)

X_p   = x_scaler.inverse_transform(X_n).astype(np.float32)
X_hat = invert_log1p(X_p, x_logmask)


# X_hat = np.clip(X_hat, 0.0, None)

df_syn = df_test.copy()
for j, col in enumerate(x_cols):
    df_syn[col] = X_hat[:, j]

df_syn = df_syn[orig_cols]
df_syn.to_csv(OUT_SYNTH, index=False)

print("Saved:", OUT_SYNTH)
print("Rows match   :", len(df_syn) == len(df_test))
print("Columns match:", list(df_syn.columns) == orig_cols)


with torch.no_grad():
    y0 = C_t[:1]

    z5 = torch.randn((5, d_x), device=device, dtype=torch.float32)
    y5 = y0.repeat(5, 1)
    x5, _ = flow._transform.inverse(z5, context=y5)

    reps = x5.detach().cpu().numpy()
    print("Std across 5 samples for same condition (standardized x-space):", reps.std(axis=0))

Device: cuda
Condition cols: ['Frequency (Hz)', 'Storage modulus (Pa)', 'Loss modulus (Pa)']
Generated cols : ['Acylamide Conc. %', 'Bis-acrylamide conc %', 'Photo-initiator conc. %', 'Layer Height. (micron)', 'Bottom Layer exposure time (s) ', 'Exposure time (s)']


/usr/local/lib/python3.12/dist-packages/nflows/transforms/lu.py:80: UserWarning: torch.triangular_solve is deprecated in favor of torch.linalg.solve_triangularand will be removed in a future PyTorch release.
torch.linalg.solve_triangular has its arguments reversed and does not return a copy of one of the inputs.
X = torch.triangular_solve(B, A).solution
should be replaced with
X = torch.linalg.solve_triangular(A, B). (Triggered internally at /pytorch/aten/src/ATen/native/BatchLinearAlgebra.cpp:2264.)
  outputs, _ = torch.triangular_solve(


Saved: /content/synth.csv
Rows match   : True
Columns match: True
Std across 5 samples for same condition (standardized x-space): [0.6606283  0.01563308 0.00877561 0.7059173  0.00473731 1.157189  ]
